# Iran Event Research Notebook

This notebook combines the main pieces of the event research workflow in one place:
- load saved limited order book snapshots from SQLite
- build a clean 5-level LOB panel for one token
- calculate a compact baseline feature set
- study sensible targets when the mid price is sticky
- load the downloaded trade history for the full event
- plot price history for every outcome from scratch


## 1. Open The Research Inputs

The notebook uses the existing order book archive and the already downloaded trade history for the event.


In [11]:
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('ggplot')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', None)  # Show full content, even for long token_id


def display(obj):
    print(obj)

SOURCE_URL = 'https://polymarket.com/event/iran-x-israelus-conflict-ends-by'


def resolve_existing_path(*candidates: str) -> Path:
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    raise FileNotFoundError(f'None of the candidate paths exist: {candidates}')


DB_PATH = Path('../db/iran_conflict_orderbooks.sqlite')
TRADES_PARQUET_PATH = resolve_existing_path(
    'cached_data/processed/iran_conflict_event_trades.parquet',
    '../cached_cached_data/processed/iran_conflict_event_trades.parquet',
)
TRADES_CSV_PATH = Path(str(TRADES_PARQUET_PATH).replace('.parquet', '.csv'))
MAX_LEVELS = 5

MARKET_SLUG = 'iran-x-israelus-conflict-ends-by-april-15'
OUTCOME_NAME = 'Yes'
TOKEN_ID = None

LOB_MOVE_IN_TICKS = 2
TARGET_HORIZONS_HOURS = (1, 2, 4, 6)

assert DB_PATH.exists(), DB_PATH
assert TRADES_PARQUET_PATH.exists() or TRADES_CSV_PATH.exists(), 'Trade history file is missing.'

con = sqlite3.connect(DB_PATH)


## 2. Helper Functions

These helpers keep the analysis cells short and make the notebook easier to reuse for another token inside the same event.


In [8]:
def query_df(sql: str, params: tuple = ()) -> pd.DataFrame:
    return pd.read_sql_query(sql, con, params=params)


def list_event_tokens(source_url: str) -> pd.DataFrame:
    sql = '''
        select
            token_id,
            condition_id,
            market_slug,
            market_question,
            group_item_title,
            outcome_name,
            outcome_index,
            updated_at_utc
        from market_outcomes
        where source_url = ?
        order by market_slug, outcome_index
    '''
    return query_df(sql, (source_url,))


def resolve_token_id(
    source_url: str,
    token_id: str | None = None,
    market_slug: str | None = None,
    outcome_name: str | None = None,
) -> str:
    tokens = list_event_tokens(source_url)
    if token_id is not None:
        token_id = str(token_id)
        matches = tokens.loc[tokens['token_id'].astype(str) == token_id]
        if matches.empty:
            raise ValueError(f'TOKEN_ID={token_id} is not present for this source.')
        return token_id

    filtered = tokens.copy()
    if market_slug is not None:
        filtered = filtered.loc[filtered['market_slug'] == market_slug]
    if outcome_name is not None:
        filtered = filtered.loc[filtered['outcome_name'] == outcome_name]

    if len(filtered) != 1:
        raise ValueError(
            'Expected exactly one token after filtering. '
            f'Found {len(filtered)} rows for market_slug={market_slug!r}, outcome_name={outcome_name!r}.'
        )
    return str(filtered.iloc[0]['token_id'])


def load_token_levels_ts(token_id: str, max_levels: int = 5) -> pd.DataFrame:
    sql = '''
        select
            s.captured_at_utc,
            s.token_id,
            s.condition_id,
            s.market_slug,
            s.outcome_name,
            l.side,
            l.level_index,
            l.price,
            l.size
        from orderbook_snapshots s
        join orderbook_levels l
          on l.snapshot_id = s.id
        where s.token_id = ?
          and l.level_index < ?
        order by s.captured_at_utc, l.side, l.level_index
    '''
    df = query_df(sql, (str(token_id), int(max_levels))).copy()
    df['captured_at_utc'] = pd.to_datetime(df['captured_at_utc'], utc=True)
    return df


def build_lob_panel(levels_df: pd.DataFrame, max_levels: int = 5) -> pd.DataFrame:
    id_cols = ['captured_at_utc', 'token_id', 'condition_id', 'market_slug', 'outcome_name']

    price_wide = (
        levels_df.pivot_table(
            index=id_cols,
            columns=['side', 'level_index'],
            values='price',
            aggfunc='first',
        )
        .sort_index(axis=1)
        .reset_index()
    )
    size_wide = (
        levels_df.pivot_table(
            index=id_cols,
            columns=['side', 'level_index'],
            values='size',
            aggfunc='first',
        )
        .sort_index(axis=1)
        .reset_index()
    )

    price_wide.columns = [
        '_'.join(str(part) for part in col if part != '') if isinstance(col, tuple) else str(col)
        for col in price_wide.columns
    ]
    size_wide.columns = [
        '_'.join(str(part) for part in col if part != '') if isinstance(col, tuple) else str(col)
        for col in size_wide.columns
    ]

    rename_price = {f'bid_{i}': f'price_bid_{i}' for i in range(max_levels)}
    rename_price |= {f'ask_{i}': f'price_ask_{i}' for i in range(max_levels)}
    rename_size = {f'bid_{i}': f'size_bid_{i}' for i in range(max_levels)}
    rename_size |= {f'ask_{i}': f'size_ask_{i}' for i in range(max_levels)}

    price_wide = price_wide.rename(columns=rename_price)
    size_wide = size_wide.rename(columns=rename_size)

    panel = price_wide.merge(size_wide, on=id_cols, how='outer').sort_values('captured_at_utc').reset_index(drop=True)

    for side in ('bid', 'ask'):
        for level in range(max_levels):
            for prefix in ('price', 'size'):
                col = f'{prefix}_{side}_{level}'
                if col not in panel.columns:
                    panel[col] = np.nan

    ordered_cols = id_cols + [
        f'{prefix}_{side}_{level}'
        for side in ('bid', 'ask')
        for prefix in ('price', 'size')
        for level in range(max_levels)
    ]
    return panel[ordered_cols].copy()


def _safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    denominator = denominator.replace(0, np.nan)
    return numerator / denominator


def _weighted_average_price(df: pd.DataFrame, side: str, levels: int) -> pd.Series:
    price_cols = [f'price_{side}_{level}' for level in range(levels)]
    size_cols = [f'size_{side}_{level}' for level in range(levels)]
    notional = sum(df[p] * df[s] for p, s in zip(price_cols, size_cols))
    depth = sum(df[s] for s in size_cols)
    return _safe_divide(notional, depth)


def build_lob_features(lob_panel: pd.DataFrame) -> pd.DataFrame:
    df = lob_panel.sort_values('captured_at_utc').reset_index(drop=True).copy()
    df['captured_at_utc'] = pd.to_datetime(df['captured_at_utc'], utc=True)

    df['best_bid'] = df['price_bid_0']
    df['best_ask'] = df['price_ask_0']
    df['spread'] = df['best_ask'] - df['best_bid']
    df['mid_price'] = (df['best_bid'] + df['best_ask']) / 2.0
    df['relative_spread'] = _safe_divide(df['spread'], df['mid_price'])
    df['microprice'] = _safe_divide(
        df['best_ask'] * df['size_bid_0'] + df['best_bid'] * df['size_ask_0'],
        df['size_bid_0'] + df['size_ask_0'],
    )
    df['top_level_imbalance'] = _safe_divide(
        df['size_bid_0'] - df['size_ask_0'],
        df['size_bid_0'] + df['size_ask_0'],
    )

    for levels in (1, 3, 5):
        bid_size_cols = [f'size_bid_{level}' for level in range(levels)]
        ask_size_cols = [f'size_ask_{level}' for level in range(levels)]
        df[f'bid_depth_{levels}'] = df[bid_size_cols].sum(axis=1, min_count=1)
        df[f'ask_depth_{levels}'] = df[ask_size_cols].sum(axis=1, min_count=1)
        df[f'depth_imbalance_{levels}'] = _safe_divide(
            df[f'bid_depth_{levels}'] - df[f'ask_depth_{levels}'],
            df[f'bid_depth_{levels}'] + df[f'ask_depth_{levels}'],
        )
        df[f'bid_wap_{levels}'] = _weighted_average_price(df, 'bid', levels)
        df[f'ask_wap_{levels}'] = _weighted_average_price(df, 'ask', levels)

    df['mid_price_change'] = df['mid_price'].diff()
    df['best_bid_change'] = df['best_bid'].diff()
    df['best_ask_change'] = df['best_ask'].diff()
    df['microprice_change'] = df['microprice'].diff()
    df['mid_return'] = _safe_divide(df['mid_price_change'], df['mid_price'].shift(1))

    book_cols = [
        col for col in df.columns
        if col.startswith('price_') or col.startswith('size_')
    ]
    df['book_changed_any'] = df[book_cols].ne(df[book_cols].shift()).any(axis=1).astype(int)
    if not df.empty:
        df.loc[df.index[0], 'book_changed_any'] = 0

    return df


def infer_tick_size(lob_features: pd.DataFrame) -> float:
    price_cols = [col for col in lob_features.columns if col.startswith('price_')]
    price_values = pd.concat([lob_features[col] for col in price_cols], ignore_index=True).dropna().sort_values().unique()
    if len(price_values) < 2:
        return 0.01
    diffs = np.diff(price_values)
    diffs = diffs[diffs > 0]
    if len(diffs) == 0:
        return 0.01
    return float(np.min(diffs))


def build_repricing_targets(
    lob_features: pd.DataFrame,
    move_in_ticks: int = 2,
    horizons_hours: tuple[int, ...] = (1, 2, 4, 6),
) -> pd.DataFrame:
    df = lob_features.sort_values('captured_at_utc').reset_index(drop=True).copy()
    tick_size = infer_tick_size(df)
    threshold = move_in_ticks * tick_size

    times = pd.to_datetime(df['captured_at_utc'], utc=True)
    mids = df['mid_price'].to_numpy(dtype=float)

    first_hit_hours = []
    first_hit_mid = []
    first_hit_delta = []

    for i in range(len(df)):
        current_mid = mids[i]
        future_mids = mids[i + 1:]
        future_times = times.iloc[i + 1:]

        if len(future_mids) == 0 or np.isnan(current_mid):
            first_hit_hours.append(np.nan)
            first_hit_mid.append(np.nan)
            first_hit_delta.append(np.nan)
            continue

        hits = np.where(np.abs(future_mids - current_mid) >= threshold)[0]
        if len(hits) == 0:
            first_hit_hours.append(np.nan)
            first_hit_mid.append(np.nan)
            first_hit_delta.append(np.nan)
            continue

        first_idx = int(hits[0])
        hit_mid = future_mids[first_idx]
        hit_time = future_times.iloc[first_idx]
        delta_hours = (hit_time - times.iloc[i]).total_seconds() / 3600.0

        first_hit_hours.append(delta_hours)
        first_hit_mid.append(hit_mid)
        first_hit_delta.append(hit_mid - current_mid)

    df['tick_size'] = tick_size
    df['repricing_threshold'] = threshold
    df['first_hit_mid_price'] = first_hit_mid
    df['first_hit_mid_delta'] = first_hit_delta
    df['time_to_repricing_hours'] = first_hit_hours
    df['repricing_observed'] = df['time_to_repricing_hours'].notna().astype(int)

    for horizon in horizons_hours:
        df[f'jump_within_{horizon}h'] = (
            df['time_to_repricing_hours'].notna() & (df['time_to_repricing_hours'] <= horizon)
        ).astype(int)

    return df


def load_event_trades() -> pd.DataFrame:
    if TRADES_PARQUET_PATH.exists():
        df = pd.read_parquet(TRADES_PARQUET_PATH)
    else:
        df = pd.read_csv(TRADES_CSV_PATH)

    df = df.copy()
    df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'], utc=True)
    return df.sort_values(['market_slug', 'outcome', 'timestamp_utc', 'transaction_hash']).reset_index(drop=True)


## 3. Inspect The Event Slice In The LOB Archive

Start with the event-level mapping so the token choice is explicit instead of hidden in a hardcoded id.


In [12]:
event_tokens_df = list_event_tokens(SOURCE_URL)
display(event_tokens_df)


                                                                          token_id                                                        condition_id  \
0    91851388830923641218615793149765322805282194001900444983744404881757241655792  0x5c2e6aef8af5931e9bfa3750364626d754531d2fada2885d45c356b175962a25   
1    42541597022580268380579424081239402820400625604765321493406160729090848893639  0x5c2e6aef8af5931e9bfa3750364626d754531d2fada2885d45c356b175962a25   
2   103971336418419351548990142781195320713490282483637854831265186666012554199721  0xa6ddb7146f48a12dbf73456d654211b01d7493829932c31b7fe85d82120d338f   
3    96397081496122471303936256377287180706458717603115171106140126963333785784378  0xa6ddb7146f48a12dbf73456d654211b01d7493829932c31b7fe85d82120d338f   
4    30474492153316523915604855839063847245493287455989692695957038531302522868025  0x23408daff185f6c89d91bd279ce97142c573454277d72ae882a9d9fd1ae79af4   
5    72672649490627178259809292941952979051200271014678303501403336114251236

## 4. Choose The Token To Study

The notebook defaults to the March 31 `Yes` token, but the selection can be changed by editing `MARKET_SLUG`, `OUTCOME_NAME`, or `TOKEN_ID` in the config cell.


In [14]:
TOKEN_ID = "91851388830923641218615793149765322805282194001900444983744404881757241655792"
print(TOKEN_ID, MARKET_SLUG, OUTCOME_NAME)
resolved_token_id = resolve_token_id(
    source_url=SOURCE_URL,
    token_id=TOKEN_ID,
    market_slug=MARKET_SLUG,
    outcome_name=OUTCOME_NAME,
)

selected_token_df = event_tokens_df.loc[event_tokens_df['token_id'].astype(str) == resolved_token_id].copy()
display(selected_token_df)


91851388830923641218615793149765322805282194001900444983744404881757241655792 iran-x-israelus-conflict-ends-by-april-15 Yes
                                                                        token_id                                                        condition_id                                    market_slug  \
0  91851388830923641218615793149765322805282194001900444983744404881757241655792  0x5c2e6aef8af5931e9bfa3750364626d754531d2fada2885d45c356b175962a25  iran-x-israelus-conflict-ends-by-april-15-618   

                               market_question group_item_title outcome_name  outcome_index                    updated_at_utc  
0  Iran x Israel/US conflict ends by April 15?         April 15          Yes              0  2026-03-17T18:45:07.321666+00:00  


## 5. Load The Saved 5-Level LOB History

This is the raw long-form snapshot history for the selected token.


In [16]:
levels_ts_df = load_token_levels_ts(resolved_token_id, max_levels=MAX_LEVELS)
levels_ts_df.head(5)


## 6. Build The Canonical 5-Level LOB Panel

A single row now represents one saved timestamp with bid and ask prices and sizes at levels `0..4`.


In [17]:
lob_panel_df = build_lob_panel(levels_ts_df, max_levels=MAX_LEVELS)
lob_panel_df.head()


## 7. Build A Compact Baseline Feature Set

These are standard limited-order-book features that are still useful even when the price grid is coarse.


In [18]:
lob_features_df = build_lob_features(lob_panel_df)
lob_features_df.head()


## 8. Check How Sticky The Mid Price Really Is

Before choosing a target, confirm whether the mid price is nearly constant and whether the book still changes underneath it.


In [19]:
mid_price_counts_df = (
    lob_features_df['mid_price']
    .value_counts(dropna=False)
    .rename_axis('mid_price')
    .reset_index(name='observations')
)

book_change_summary_df = pd.DataFrame([
    {
        'n_snapshots': len(lob_features_df),
        'book_changed_share': float(lob_features_df['book_changed_any'].mean()),
        'mid_price_change_share': float(lob_features_df['mid_price_change'].ne(0).fillna(False).mean()),
        'spread_change_share': float(lob_features_df['spread'].diff().ne(0).fillna(False).mean()),
        'top_level_imbalance_std': float(lob_features_df['top_level_imbalance'].std()),
    }
])

display(mid_price_counts_df)
display(book_change_summary_df)


   mid_price  observations
0      0.245          7329
1      0.240            10
   n_snapshots  book_changed_share  mid_price_change_share  spread_change_share  top_level_imbalance_std
0         7339            0.560294                0.000409             0.000409                  0.24838


In [20]:
plot_df = lob_features_df.sort_values('captured_at_utc').copy()
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True, constrained_layout=True)

axes[0].plot(plot_df['captured_at_utc'], plot_df['mid_price'], color='#1f77b4', linewidth=1.2)
axes[0].set_title('Selected Token Mid Price')
axes[0].set_ylabel('Mid price')

axes[1].plot(plot_df['captured_at_utc'], plot_df['spread'], color='#d62728', linewidth=1.2)
axes[1].set_title('Spread')
axes[1].set_ylabel('Spread')

axes[2].plot(plot_df['captured_at_utc'], plot_df['top_level_imbalance'], color='#2ca02c', linewidth=1.2)
axes[2].set_title('Top-Level Imbalance')
axes[2].set_ylabel('Imbalance')
axes[2].set_xlabel('Captured at UTC')

plt.show()


## 9. Analyze Candidate Targets For A Sticky Mid Price

When the mid price barely moves, next-step regression is usually a weak problem statement. Event-style targets are more natural:
- repricing within a fixed horizon
- time to repricing
- future book activity instead of future price level
- regime changes from stale to active periods


In [21]:
repricing_target_df = build_repricing_targets(
    lob_features_df,
    move_in_ticks=LOB_MOVE_IN_TICKS,
    horizons_hours=TARGET_HORIZONS_HOURS,
)

target_rate_df = pd.DataFrame([
    {
        'tick_size': float(repricing_target_df['tick_size'].iloc[0]),
        'repricing_threshold': float(repricing_target_df['repricing_threshold'].iloc[0]),
        **{f'jump_rate_{h}h': float(repricing_target_df[f'jump_within_{h}h'].mean()) for h in TARGET_HORIZONS_HOURS},
        'repricing_observed_rate': float(repricing_target_df['repricing_observed'].mean()),
        'median_time_to_repricing_hours': float(repricing_target_df['time_to_repricing_hours'].dropna().median()) if repricing_target_df['time_to_repricing_hours'].notna().any() else np.nan,
    }
])

target_design_df = pd.DataFrame([
    {
        'target_family': 'jump prediction',
        'example_label': 'jump_within_4h',
        'why_it_is_sensible': 'Useful when the price sits on long plateaus and reprices only occasionally.',
        'model_type': 'binary classification',
    },
    {
        'target_family': 'time to repricing',
        'example_label': 'time_to_repricing_hours',
        'why_it_is_sensible': 'Matches waiting-time dynamics better than next-step direction forecasting.',
        'model_type': 'survival or duration model',
    },
    {
        'target_family': 'book activity',
        'example_label': 'book_changed_any over the next window',
        'why_it_is_sensible': 'The book can be informative even when the midpoint is nearly flat.',
        'model_type': 'binary classification or count model',
    },
    {
        'target_family': 'regime shift',
        'example_label': 'active_regime_next_1h',
        'why_it_is_sensible': 'Separates stale periods from active periods without forcing a price move label.',
        'model_type': 'classification',
    },
])

display(target_rate_df)
display(target_design_df)


   tick_size  repricing_threshold  jump_rate_1h  jump_rate_2h  jump_rate_4h  jump_rate_6h  repricing_observed_rate  median_time_to_repricing_hours
0       0.01                 0.02           0.0           0.0           0.0           0.0                      0.0                             NaN
       target_family                          example_label                                                               why_it_is_sensible                            model_type
0    jump prediction                         jump_within_4h      Useful when the price sits on long plateaus and reprices only occasionally.                 binary classification
1  time to repricing                time_to_repricing_hours       Matches waiting-time dynamics better than next-step direction forecasting.            survival or duration model
2      book activity  book_changed_any over the next window               The book can be informative even when the midpoint is nearly flat.  binary classification or co

In [22]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True, constrained_layout=True)

axes[0].plot(repricing_target_df['captured_at_utc'], repricing_target_df['mid_price'], color='#1f77b4', linewidth=1.2)
axes[0].set_title('Mid Price Across Saved Snapshots')
axes[0].set_ylabel('Mid price')

axes[1].plot(
    repricing_target_df['captured_at_utc'],
    repricing_target_df['time_to_repricing_hours'],
    color='#ff7f0e',
    linewidth=1.0,
)
axes[1].set_title(f'Time To Repricing (threshold = {LOB_MOVE_IN_TICKS} ticks)')
axes[1].set_ylabel('Hours')
axes[1].set_xlabel('Captured at UTC')

plt.show()


## 10. Load The Downloaded Trade History

The trade history was downloaded earlier for the whole event. Here it is loaded from disk and summarized before plotting.


In [ ]:
trades_df = load_event_trades()
trade_summary_df = (
    trades_df.groupby(['market_slug', 'outcome'], as_index=False)
    .agg(
        n_trades=('transaction_hash', 'size'),
        first_trade_utc=('timestamp_utc', 'min'),
        last_trade_utc=('timestamp_utc', 'max'),
        total_size=('size', 'sum'),
        median_price=('price', 'median'),
    )
    .sort_values(['market_slug', 'outcome'])
    .reset_index(drop=True)
)

display(trades_df.head())
display(trade_summary_df)

              timestamp_utc  price        size outcome                                                    transaction_hash                                    market_slug  \
0 2026-03-12 19:36:59+00:00   0.70  600.000000      No  0x1d02c275d96a9a9dac7981e787af7704f1b8b034c0bf72db0bb1b6009e9693e1  iran-x-israelus-conflict-ends-by-april-15-618   
1 2026-03-12 21:17:05+00:00   0.66   50.000000      No  0xa3b46ece60d5897fecca0dd5f738768af89c23cd3cdcd46a520698cc25718885  iran-x-israelus-conflict-ends-by-april-15-618   
2 2026-03-12 21:17:05+00:00   0.63   32.000000      No  0xa3b46ece60d5897fecca0dd5f738768af89c23cd3cdcd46a520698cc25718885  iran-x-israelus-conflict-ends-by-april-15-618   
3 2026-03-12 21:17:05+00:00   0.64   50.000000      No  0xa3b46ece60d5897fecca0dd5f738768af89c23cd3cdcd46a520698cc25718885  iran-x-israelus-conflict-ends-by-april-15-618   
4 2026-03-12 21:17:05+00:00   0.63   49.864863      No  0xa3b46ece60d5897fecca0dd5f738768af89c23cd3cdcd46a520698cc25718885  iran-x-isra

## 11. Plot Price History For Every Outcome From Scratch

Each subplot shows one market in the event, with separate trade-price paths for `Yes` and `No`.


In [24]:
market_order = (
    trades_df.groupby('market_slug')['transaction_hash']
    .size()
    .sort_values(ascending=False)
    .index
    .tolist()
)

question_map = (
    trades_df[['market_slug', 'market_question']]
    .drop_duplicates()
    .set_index('market_slug')['market_question']
    .to_dict()
)

fig, axes = plt.subplots(
    len(market_order),
    1,
    figsize=(15, max(3.0 * len(market_order), 10)),
    sharex=True,
    constrained_layout=True,
)

if len(market_order) == 1:
    axes = [axes]

colors = {'Yes': '#2ca25f', 'No': '#de2d26'}

for ax, market_slug in zip(axes, market_order):
    market_trades = trades_df.loc[trades_df['market_slug'] == market_slug].sort_values('timestamp_utc')
    question = question_map.get(market_slug, market_slug)

    for outcome in ('Yes', 'No'):
        outcome_trades = market_trades.loc[market_trades['outcome'] == outcome]
        if outcome_trades.empty:
            continue
        ax.plot(
            outcome_trades['timestamp_utc'],
            outcome_trades['price'],
            label=outcome,
            color=colors[outcome],
            linewidth=1.0,
            alpha=0.9,
        )

    ax.set_title(question)
    ax.set_ylabel('Trade price')
    ax.set_ylim(-0.02, 1.02)
    ax.legend(loc='upper right')

axes[-1].set_xlabel('Trade timestamp UTC')
plt.show()


## 12. Optional Cross-Check: Selected Token Versus Its Trade Path

This last view narrows the trade history back to the chosen market and outcome so the LOB slice and the trade tape can be inspected together.


In [25]:
selected_market_slug = selected_token_df.iloc[0]['market_slug']
selected_outcome_name = selected_token_df.iloc[0]['outcome_name']
selected_trades_df = trades_df.loc[
    (trades_df['market_slug'] == selected_market_slug)
    & (trades_df['outcome'] == selected_outcome_name)
].sort_values('timestamp_utc')

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False, constrained_layout=True)

axes[0].plot(lob_features_df['captured_at_utc'], lob_features_df['mid_price'], color='#1f77b4', linewidth=1.2)
axes[0].set_title('Selected Token Mid Price From Saved LOB Snapshots')
axes[0].set_ylabel('Mid price')

axes[1].plot(selected_trades_df['timestamp_utc'], selected_trades_df['price'], color='#2ca25f', linewidth=1.1)
axes[1].set_title('Selected Outcome Trade Price History')
axes[1].set_ylabel('Trade price')
axes[1].set_xlabel('Timestamp UTC')
axes[1].set_ylim(-0.02, 1.02)

plt.show()


## 13. Pull Iran-Relevant News From GDELT For The Last Week

This final section fetches recent Iran-related coverage from the GDELT DOC API for the exact window `2026-03-11` through `2026-03-18`.

GDELT sometimes returns a plain-text rate-limit message instead of JSON. If that happens, wait at least 5 seconds and rerun the cell below.


In [28]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "clients").exists() else cwd.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

Repo root: /Users/sneddy/research/polymarket_research


In [ ]:
from datetime import UTC, datetime
import time

from collectors.news_collector import NewsCollector

RAW_GDELT_QUERY = '(iran OR tehran OR iranian OR oil or hormuz or uae or dubai or abu-dhabi or saudy)'
GDELT_QUERY = RAW_GDELT_QUERY.replace('abu-dhabi', '"abu-dhabi"')
GDELT_START = datetime(2026, 3, 11, 0, 0, 0, tzinfo=UTC)
GDELT_END = datetime(2026, 3, 18, 23, 59, 59, tzinfo=UTC)
GDELT_MAX_RECORDS_PER_DAY = 25
GDELT_SLEEP_SECONDS = 6.0

news = NewsCollector()
window_starts = pd.date_range(GDELT_START, GDELT_END + pd.Timedelta(days=1), freq='1D', tz='UTC')
gdelt_chunks = []
gdelt_window_stats = []

for idx in range(len(window_starts) - 1):
    window_start = window_starts[idx]
    window_end = min(window_starts[idx + 1] - pd.Timedelta(seconds=1), pd.Timestamp(GDELT_END))

    chunk_df = news.search_gdelt(
        query=GDELT_QUERY,
        start_date=window_start.to_pydatetime(),
        end_date=window_end.to_pydatetime(),
        language='english',
        dedupe=False,
        frame_type='pandas',
        max_records=GDELT_MAX_RECORDS_PER_DAY,
        page_size=GDELT_MAX_RECORDS_PER_DAY,
        show_progress=False,
    )

    chunk_df = chunk_df.copy()
    chunk_df['window_start_utc'] = window_start
    chunk_df['window_end_utc'] = window_end
    gdelt_chunks.append(chunk_df)
    gdelt_window_stats.append({
        'window_start_utc': window_start,
        'window_end_utc': window_end,
        'rows': len(chunk_df),
    })

    if idx < len(window_starts) - 2:
        time.sleep(GDELT_SLEEP_SECONDS)

gdelt_news_df = pd.concat(gdelt_chunks, ignore_index=True) if gdelt_chunks else pd.DataFrame()
if not gdelt_news_df.empty:
    gdelt_news_df = (
        gdelt_news_df
        .sort_values(['timestamp_utc', 'domain', 'title'], ascending=[False, True, True])
        .drop_duplicates(subset=['url', 'title'], keep='first')
        .reset_index(drop=True)
    )

gdelt_window_stats_df = pd.DataFrame(gdelt_window_stats)

display(pd.DataFrame([{
    'raw_query': RAW_GDELT_QUERY,
    'effective_query': GDELT_QUERY,
    'start_utc': GDELT_START.isoformat(),
    'end_utc': GDELT_END.isoformat(),
    'language': 'english',
    'dedupe_after_concat': True,
    'max_records_per_day': GDELT_MAX_RECORDS_PER_DAY,
    'window_count': len(gdelt_window_stats_df),
    'returned_articles': len(gdelt_news_df),
}]))

display(gdelt_window_stats_df)

gdelt_news_view_df = gdelt_news_df.loc[:, [
    'timestamp_utc',
    'title',
    'source',
    'domain',
    'language',
    'url',
]] if not gdelt_news_df.empty else pd.DataFrame(columns=['timestamp_utc', 'title', 'source', 'domain', 'language', 'url'])

display(gdelt_news_view_df)

if not gdelt_news_df.empty:
    daily_counts_df = (
        gdelt_news_df.assign(publish_day=gdelt_news_df['timestamp_utc'].dt.floor('D'))
        .groupby('publish_day', as_index=False)
        .agg(article_count=('title', 'size'))
    )

    fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=True)
    ax.plot(daily_counts_df['publish_day'], daily_counts_df['article_count'], marker='o', linewidth=1.4)
    ax.set_title('GDELT Iran-Relevant Article Count By Day')
    ax.set_xlabel('Publish day UTC')
    ax.set_ylabel('Article count')
    plt.show()
